# FarmTech Solutions — Entrega 2 (Fase 6)
**Aluno:** Diego Filipe Pereira de Araujo — RM567064

**Ambiente:** **Google Colab** (GPU opcional).

**Objetivo:** comparar **três abordagens** sobre a mesma base da Entrega 1:
1. **YOLO custom** (`fase6_ep60` da E1).
2. **YOLO padrão** (`yolov5s.pt` COCO).
3. **CNN do zero** (TensorFlow — **objeto_a** × **objeto_b**).

**Importante:** na raiz do projeto (`PROJECT`) precisam existir a pasta **`runs/`** (com `train/fase6_ep60/weights/best.pt`) e o arquivo **`data_fase6.yaml`**. No Colab, use a **mesma sessão** após a E1 **ou** faça upload de `runs` e `data_fase6.yaml` para `/content/farmtech/`.


In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

PROJECT = Path("/content/farmtech")
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT = Path("/content/drive/MyDrive/SEU_CAMINHO/FIAP-ON-AI/Fase 6/Capitulo 1")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "assets").mkdir(parents=True, exist_ok=True)

DATA_YOLO = PROJECT / "assets" / "dataset_yolo_farmtech"
ZIP_ALVO = PROJECT / "dataset_yolo_farmtech.zip"
if ZIP_ALVO.is_file() and not DATA_YOLO.is_dir():
    with zipfile.ZipFile(ZIP_ALVO, "r") as zf:
        zf.extractall(PROJECT / "assets")

DATA_CNN = PROJECT / "assets" / "cnn_classificacao_farmtech"

if not DATA_YOLO.is_dir():
    raise FileNotFoundError(
        f"Dataset não encontrado: {DATA_YOLO}. Envie dataset_yolo_farmtech.zip para /content/farmtech/ "
        "ou use USE_GOOGLE_DRIVE=True."
    )

YOLO_DIR = PROJECT / "yolov5"
if not YOLO_DIR.is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLO_DIR)],
        check=True,
        cwd=str(PROJECT),
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(YOLO_DIR / "requirements.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"], check=True)

for sp in ("train", "valid"):
    for cls in ("objeto_a", "objeto_b"):
        (DATA_CNN / sp / cls).mkdir(parents=True, exist_ok=True)

for sp in ("train", "valid"):
    src_im = DATA_YOLO / sp / "images"
    for cls in ("objeto_a", "objeto_b"):
        for p in sorted(src_im.glob(cls + "_*")):
            if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp", ".bmp"):
                shutil.copy2(p, DATA_CNN / sp / cls / p.name)

WEIGHTS_CUSTOM = PROJECT / "runs" / "train" / "fase6_ep60" / "weights" / "best.pt"
DATA_YAML = PROJECT / "data_fase6.yaml"
TEST_IMAGES = DATA_YOLO / "test" / "images"

os.chdir(YOLO_DIR)

METRICAS = {}

print("PROJECT:", PROJECT)
print("DATA_YOLO:", DATA_YOLO)
print("DATA_CNN:", DATA_CNN)
print("WEIGHTS_CUSTOM:", WEIGHTS_CUSTOM, "| existe:", WEIGHTS_CUSTOM.is_file())
print("DATA_YAML:", DATA_YAML, "| existe:", DATA_YAML.is_file())


## 1. YOLO custom (Entrega 1)
Validação com `val.py` no conjunto de validação do YAML. **Copie** os valores de **mAP@0.5** (e precisão/recall se quiser) da saída abaixo para o quadro comparativo final.


In [ ]:
import time
from pathlib import Path

assert WEIGHTS_CUSTOM.is_file(), "Treine a E1 antes ou ajuste WEIGHTS_CUSTOM."
assert DATA_YAML.is_file(), "Coloque data_fase6.yaml na raiz do projeto (mesma pasta que runs/)."

!WANDB_MODE=disabled python3 val.py --weights {WEIGHTS_CUSTOM} --data {DATA_YAML} --img 640


In [ ]:
# Tempo de inferência no conjunto de teste (YOLO custom)
import time

t0 = time.perf_counter()
!WANDB_MODE=disabled python3 detect.py --weights {WEIGHTS_CUSTOM} --img 640 --conf 0.25 --source {TEST_IMAGES} --project ../runs/detect --name infer_custom_e2 --exist-ok
t1 = time.perf_counter()

imgs = [p for p in Path(TEST_IMAGES).glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
METRICAS["yolo_custom_infer_s"] = t1 - t0
METRICAS["yolo_custom_n_img"] = len(imgs)
print(f"YOLO custom — inferência: {METRICAS['yolo_custom_infer_s']:.2f} s | {len(imgs)} imagens")
if imgs:
    print(f"Média ~{(t1 - t0) / len(imgs) * 1000:.1f} ms/imagem")

## 2. YOLO padrão (pré-treinada COCO)
Mesmo comando de inferência, pesos **yolov5s.pt** (80 classes COCO). No nosso problema (maçã/laranja), as classes **não batem** — serve para comparar **facilidade** (zero treino) e **tempo**, e discutir **precisão** no nosso domínio.


In [ ]:
import time

t0 = time.perf_counter()
!WANDB_MODE=disabled python3 detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source {TEST_IMAGES} --project ../runs/detect --name infer_yolov5_coco --exist-ok
t1 = time.perf_counter()

imgs = [p for p in Path(TEST_IMAGES).glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
METRICAS["yolo_coco_infer_s"] = t1 - t0
METRICAS["yolo_coco_n_img"] = len(imgs)
print(f"YOLO COCO — inferência: {METRICAS['yolo_coco_infer_s']:.2f} s | {len(imgs)} imagens")
if imgs:
    print(f"Média ~{(t1 - t0) / len(imgs) * 1000:.1f} ms/imagem")

## 3. CNN do zero (classificação)
Rede pequena em Keras; entrada organizada por pastas (`objeto_a` / `objeto_b`). Classifica a **imagem inteira**, não desenha caixas — compare com YOLO nesse aspecto.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 128
BATCH = 16
EPOCHS = 15

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_CNN / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_CNN / "valid",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
)
print("Classes:", train_ds.class_names)

In [ ]:
num_classes = len(train_ds.class_names)
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
import time

t0 = time.perf_counter()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
t1 = time.perf_counter()
METRICAS["cnn_treino_s"] = t1 - t0
print(f"CNN — tempo de treino: {METRICAS['cnn_treino_s']:.1f} s")

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=0)
METRICAS["cnn_val_loss"] = float(loss)
METRICAS["cnn_val_acc"] = float(acc)
print(f"CNN — validação: loss={loss:.4f}  accuracy={acc:.4f}")

In [ ]:
import time

t0 = time.perf_counter()
_ = model.predict(val_ds, verbose=0)
t1 = time.perf_counter()
METRICAS["cnn_predict_val_s"] = t1 - t0
print(f"CNN — predict em todo o val: {METRICAS['cnn_predict_val_s']:.3f} s")

## 4. Análise crítica (enunciado)

Preencha **mAP do YOLO custom** com o número que apareceu na saída do `val.py` da seção 1 (ex.: `all ... 0.xxx`).

| Critério | YOLO custom (E1) | YOLO COCO | CNN do zero |
|----------|------------------|-----------|-------------|
| Facilidade / integração | Treino + YAML + rótulos; mais passos, mas resolve detecção | Só baixar pesos; não precisa treinar | Mais simples se só quiser A vs B na imagem inteira |
| Precisão no problema | Melhor para **localizar** maçã/laranja após fine-tuning | Baixa utilidade sem classes iguais às nossas | Boa para classe da foto inteira; não acha posição do objeto |
| Tempo de treino / customização | Alto (duas corridas na E1) | ~zero | Moderado (célula acima `cnn_treino_s`) |
| Tempo de inferência | Ver `METRICAS['yolo_custom_infer_s']` | Ver `METRICAS['yolo_coco_infer_s']` | Ver `METRICAS['cnn_predict_val_s']` no conjunto de validação |

**Interpretação:** O modelo **custom** é o adequado para **detecção** no dataset rotulado. O **COCO** ilustra por que modelo genérico não substitui fine-tuning. A **CNN** é útil como classificador rápido, mas não substitui caixas delimitadoras.


In [ ]:
# Resumo automático dos tempos medidos (ajuste mAP manualmente na célula Markdown acima)
from pprint import pprint

print("Tempos gravados em METRICAS:")
pprint(METRICAS)

print("\\n--- Texto para colar no relatório (revise os números de mAP na saída do val.py) ---")
print(f"""
YOLO custom: inferência no teste ≈ {METRICAS.get('yolo_custom_infer_s', 0):.2f} s para {METRICAS.get('yolo_custom_n_img', 0)} imagens.
YOLO COCO:   inferência no teste ≈ {METRICAS.get('yolo_coco_infer_s', 0):.2f} s para {METRICAS.get('yolo_coco_n_img', 0)} imagens.
CNN:         treino ≈ {METRICAS.get('cnn_treino_s', 0):.1f} s | acurácia validação ≈ {METRICAS.get('cnn_val_acc', 0):.2%}
             predict no val ≈ {METRICAS.get('cnn_predict_val_s', 0):.3f} s
""")